In [1]:
import os
import gc
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib            import Path

from collections        import defaultdict

from torch.utils.data   import (DataLoader, 
                                TensorDataset, 
                                WeightedRandomSampler)

from sklearn.metrics    import (accuracy_score,
                                confusion_matrix,
                                classification_report,
                                balanced_accuracy_score, 
                                f1_score)

import sys
sys.path.append('..')

import module

from src.model.modules    import (MicroExpressionDataset, 
                                  Normalizer, 
                                  SpatialTemporalCNN)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
torch.backends.cudnn.benchmark = True
print(f'Device: {device}')

Device: cuda


In [2]:
# ── Konfigurasi ────────────────────────────────────────────────────────
ANNOTATIONS  = os.path.join(Path.home(), "datasets", "anxiety_raw", "annotations-v2.csv")
STAGE        = 'all'
SEED         = 42

EPOCHS       = 200
LR           = 3e-4
WEIGHT_DECAY = 1e-4
BATCH_SIZE   = 32
PATIENCE     = 20    # stop: train_loss tidak turun

CKPT_DIR     = './checkpoints_lopo'
os.makedirs(CKPT_DIR, exist_ok=True)

print(f'LOPO | Strategi C | EPOCHS={EPOCHS} | PATIENCE={PATIENCE}')
print(f'Checkpoint dir: {CKPT_DIR}')

LOPO | Strategi C | EPOCHS=200 | PATIENCE=20
Checkpoint dir: ./checkpoints_lopo


In [3]:
# ── Dataset & Folds ─────────────────────────────────────────────────────
dataset = MicroExpressionDataset(
    annotations_file=ANNOTATIONS, stage=STAGE,
    mode='train', valid_only=True)

print(f'Total samples : {len(dataset)}')
print(f'Total subjek  : {dataset.annotations["subject_id"].nunique()}')
y_all = dataset.get_labels()
print(f'Label dist    : high={(y_all==1).sum()}, low={(y_all==0).sum()}')

# Cek skipped clips jika ada
if len(dataset.skipped_clips) > 0:
    print(f'\nWarning: {len(dataset.skipped_clips)} clip di-skip (file .npy tidak ada)')
    print(dataset.skipped_clips[['subject_id','clip','stage']].to_string(index=False))

folds = MicroExpressionDataset.build_lopo_folds(
    dataset.annotations, seed=SEED)
print(f'\nLOPO: {len(folds)} fold siap.')

Total samples : 559
Total subjek  : 56
Label dist    : high=265, low=294

LOPO: 24 fold siap.


In [4]:
# ── Helper functions ─────────────────────────────────────────────────────
from torch.utils.data import TensorDataset

def normalize_fold(dataset, train_idx, test_idx):
    dataset.set_mode('eval')
    dataset.clear_cache()
    X_all, y_all, subs_all = dataset.get_all_features()
    X_tr, y_tr = X_all[train_idx], y_all[train_idx]
    X_te, y_te = X_all[test_idx],  y_all[test_idx]
    norm = Normalizer()
    X_tr = norm.fit_transform(X_tr)
    X_te = norm.transform(X_te)
    return X_tr, X_te, y_tr, y_te, norm


def make_loaders(X_tr, y_tr, X_te, y_te, batch_size):
    counts  = np.bincount(y_tr, minlength=2)
    weights = torch.tensor([1./counts[l] for l in y_tr], dtype=torch.float)
    sampler = WeightedRandomSampler(weights, len(weights), replacement=True)
    tr_ds   = TensorDataset(torch.from_numpy(X_tr),
                             torch.from_numpy(y_tr).long(),
                             torch.arange(len(X_tr)))
    te_ds   = TensorDataset(torch.from_numpy(X_te),
                             torch.from_numpy(y_te).long(),
                             torch.arange(len(X_te)))
    return (DataLoader(tr_ds, batch_size=batch_size, sampler=sampler, drop_last=True),
            DataLoader(te_ds, batch_size=batch_size*2, shuffle=False))


def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total = 0.
    for feat, label, _ in loader:
        feat, label = feat.to(device), label.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            loss = criterion(model(feat), label)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        total += loss.item()
    return total / max(len(loader), 1)


def evaluate(model, loader, subj_names, verbose=False):
    model.eval()
    s_preds, s_targets = [], []
    subj_probs, subj_targets = defaultdict(list), {}
    with torch.no_grad():
        for feat, label, idx in loader:
            prob = torch.softmax(model(feat.to(device)), dim=1).cpu().numpy()
            pred = np.argmax(prob, axis=1)
            lnp  = label.numpy()
            s_preds.extend(pred.tolist())
            s_targets.extend(lnp.tolist())
            for i, si in enumerate(idx.tolist()):
                subj = subj_names[si]
                subj_probs[subj].append(prob[i])
                subj_targets[subj] = int(lnp[i])
    names   = sorted(subj_probs)
    sp      = [int(np.argmax(np.mean(subj_probs[s], axis=0))) for s in names]
    st      = [subj_targets[s] for s in names]
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        # Catatan: jika model prediksi satu kelas saja (bias fold),
        # balanced_accuracy_score tetap valid tapi f1 macro = 0.
        # Ini perilaku wajar di LOPO dengan test set sangat kecil.
        unique_pred = set(sp)
        m = {
            'sample_targets'  : s_targets,
            'sample_preds'    : s_preds,
            'sample_uar'      : balanced_accuracy_score(s_targets, s_preds),
            'subject_uar'     : balanced_accuracy_score(st, sp),
            'subject_f1'      : f1_score(st, sp, average='macro', zero_division=0),
            'subject_acc'     : accuracy_score(st, sp),
            'subject_cm'      : confusion_matrix(st, sp, labels=[0,1]),
            'subject_report'  : classification_report(
                st, sp, labels=[0,1], target_names=['low','high'],
                digits=4, zero_division=0),
            'subject_preds'   : list(zip(names, st, sp)),
            'pred_one_class'  : len(unique_pred) == 1,
        }
    if verbose:
        print(f'  sample UAR  : {m["sample_uar"]:.4f}')
        print(f'  subject UAR : {m["subject_uar"]:.4f}')
        print(f'  macro-F1    : {m["subject_f1"]:.4f}')
        print(f'  confusion   :\n{m["subject_cm"]}')
        print(f'  report      :\n{m["subject_report"]}')
    return m


print('Helpers siap.')

Helpers siap.


In [5]:
# ── LOPO Training Loop — Strategi C ─────────────────────────────────────
# Stop   : train loss tidak turun selama PATIENCE epoch
# Ckpt   : disimpan setiap val UAR membaik
# ─────────────────────────────────────────────────────────────────────────

criterion     = nn.CrossEntropyLoss(label_smoothing=0.05)
fold_results  = []
global_true   = []
global_pred   = []
best_global   = -1.

for fold_info in folds:
    fold      = fold_info['fold']
    train_idx = fold_info['train_idx']
    test_idx  = fold_info['test_idx']

    print(f'\n{"="*65}')
    print(f'Fold {fold:02d} | test={sorted(fold_info["test_subjects"])} '
          f'dist={fold_info["test_label_dist"]}')
    print(f'  train={len(train_idx)} dist={fold_info["train_label_dist"]}')

    # Normalisasi
    X_tr, X_te, y_tr, y_te, norm = normalize_fold(dataset, train_idx, test_idx)
    subs_all  = dataset.get_subject_ids()
    subs_test = [subs_all[i] for i in test_idx]
    train_loader, test_loader = make_loaders(X_tr, y_tr, X_te, y_te, BATCH_SIZE)

    # Model
    model     = SpatialTemporalCNN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR,
                                  weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS, eta_min=1e-6)
    scaler    = torch.amp.GradScaler('cuda')

    # Strategi C state
    best_loss = float('inf')
    best_uar  = -1.
    best_ckpt = None
    no_imp    = 0
    stopped_ep = EPOCHS - 1

    dataset.set_mode('train')

    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
        scheduler.step()

        m    = evaluate(model, test_loader, subs_test)
        uar  = m['subject_uar']
        lr_v = scheduler.get_last_lr()[0]

        # Checkpoint: val UAR membaik
        if uar > best_uar:
            best_uar  = uar
            best_ckpt = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        # Stop: train loss
        if loss < best_loss - 1e-4:
            best_loss = loss
            no_imp    = 0
        else:
            no_imp += 1

        if epoch % 10 == 0 or no_imp >= PATIENCE:
            print(f'  Ep {epoch:03d} | loss={loss:.4f} '
                  f'best_loss={best_loss:.4f} | '
                  f'uar={uar:.4f} best_uar={best_uar:.4f} | '
                  f'no_imp={no_imp}/{PATIENCE} | lr={lr_v:.1e}')

        if no_imp >= PATIENCE:
            stopped_ep = epoch
            print(f'  >> Early stop ep {epoch} (train_loss)')
            break

    # Load best checkpoint
    if best_ckpt:
        model.load_state_dict(best_ckpt)

    # Evaluasi final
    print(f'\nFold {fold:02d} — Final:')
    fm = evaluate(model, test_loader, subs_test, verbose=True)
    if fm.get('pred_one_class', False):
        only_cls = 'HIGH' if fm['sample_preds'][0] == 1 else 'LOW'
        print(f'  [!] Model hanya prediksi {only_cls} di fold ini '
              f'(bias fold — wajar di LOPO dengan test set kecil)')

    global_true.extend(fm['sample_targets'])
    global_pred.extend(fm['sample_preds'])

    row = {'fold': fold,
           'test_subjects': sorted(fold_info['test_subjects']),
           'uar': fm['subject_uar'], 'f1': fm['subject_f1'],
           'acc': fm['subject_acc'], 'sample_uar': fm['sample_uar'],
           'best_uar': best_uar, 'best_loss': best_loss,
           'stopped_ep': stopped_ep,
           'subject_preds': fm['subject_preds']}
    fold_results.append(row)

    # Simpan checkpoint global terbaik
    if fm['subject_uar'] > best_global:
        best_global = fm['subject_uar']
        ckpt_path = os.path.join(CKPT_DIR,
            f'lopo_fold{fold:02d}_uar{fm["subject_uar"]:.4f}.pt')
        torch.save(best_ckpt, ckpt_path)
        # Save normalizer alongside checkpoint — required for inference
        norm_path = ckpt_path.replace('.pt', '_normalizer.npz')
        norm.save(norm_path)
        print(f'  >> Checkpoint: {ckpt_path}')
        print(f'  >> Normalizer: {norm_path}')

    # Cleanup
    del model, optimizer, scheduler, scaler, best_ckpt
    del X_tr, X_te, y_tr, y_te, train_loader, test_loader
    gc.collect(); torch.cuda.empty_cache()


Fold 00 | test=['ananda_satria_putra_nugraha', 'tora_digda_kristiawan'] dist=[10 10]
  train=539 dist=[229 310]
  Ep 000 | loss=1.2973 best_loss=1.2973 | uar=0.5000 best_uar=0.5000 | no_imp=0/20 | lr=3.0e-04
  Ep 010 | loss=0.8532 best_loss=0.8532 | uar=0.5000 best_uar=1.0000 | no_imp=0/20 | lr=3.0e-04
  Ep 020 | loss=0.7921 best_loss=0.7436 | uar=1.0000 best_uar=1.0000 | no_imp=1/20 | lr=2.9e-04
  Ep 030 | loss=0.6683 best_loss=0.6679 | uar=0.5000 best_uar=1.0000 | no_imp=1/20 | lr=2.8e-04
  Ep 040 | loss=0.6818 best_loss=0.6341 | uar=1.0000 best_uar=1.0000 | no_imp=2/20 | lr=2.7e-04
  Ep 050 | loss=0.6327 best_loss=0.6193 | uar=1.0000 best_uar=1.0000 | no_imp=4/20 | lr=2.5e-04
  Ep 060 | loss=0.6243 best_loss=0.5853 | uar=1.0000 best_uar=1.0000 | no_imp=7/20 | lr=2.4e-04
  Ep 070 | loss=0.5734 best_loss=0.5664 | uar=1.0000 best_uar=1.0000 | no_imp=2/20 | lr=2.2e-04
  Ep 080 | loss=0.5476 best_loss=0.5476 | uar=1.0000 best_uar=1.0000 | no_imp=0/20 | lr=1.9e-04
  Ep 090 | loss=0.5683 

In [6]:
# ── Ringkasan LOPO ───────────────────────────────────────────────────────
print(f'\n{"="*65}')
print('RINGKASAN LOPO — Strategi C')
print(f'{"="*65}')

df = pd.DataFrame(fold_results)
uars = df['uar'].values
f1s  = df['f1'].values

print(f'Subject UAR (mean +/- std) : {uars.mean():.4f} +/- {uars.std():.4f}')
print(f'Subject F1  (mean +/- std) : {f1s.mean():.4f}  +/- {f1s.std():.4f}')

g_uar = balanced_accuracy_score(global_true, global_pred)
g_f1  = f1_score(global_true, global_pred, average='macro', zero_division=0)
g_cm  = confusion_matrix(global_true, global_pred, labels=[0,1])
print(f'Clip UAR global            : {g_uar:.4f}')
print(f'Clip F1  global            : {g_f1:.4f}')
print(f'Confusion global           : {g_cm.tolist()}')

# Bootstrap CI
boots = [np.mean(np.random.choice(uars, len(uars), replace=True))
         for _ in range(5000)]
lo, hi = np.percentile(boots, [2.5, 97.5])
print(f'Bootstrap 95% CI           : [{lo:.4f}, {hi:.4f}]')

# Perbandingan dengan GroupKFold
print(f'\nPerbandingan:')
print(f'  GroupKFold-5 Strategi C  : 0.6935 +/- 0.0880 [0.635, 0.784]')
print(f'  LOPO Strategi C          : {uars.mean():.4f} +/- {uars.std():.4f} [{lo:.3f}, {hi:.3f}]')

# Per-fold detail
print(f'\nPer-fold results:')
cols = ['fold','test_subjects','uar','f1','acc','sample_uar','stopped_ep']
print(df[cols].to_string(index=False))


RINGKASAN LOPO — Strategi C
Subject UAR (mean +/- std) : 0.8125 +/- 0.2818
Subject F1  (mean +/- std) : 0.7639  +/- 0.3399
Clip UAR global            : 0.5737
Clip F1  global            : 0.5699
Confusion global           : [[154, 120], [85, 120]]
Bootstrap 95% CI           : [0.6875, 0.9167]

Perbandingan:
  GroupKFold-5 Strategi C  : 0.6935 +/- 0.0880 [0.635, 0.784]
  LOPO Strategi C          : 0.8125 +/- 0.2818 [0.688, 0.917]

Per-fold results:
 fold                                              test_subjects  uar       f1  acc  sample_uar  stopped_ep
    0       [ananda_satria_putra_nugraha, tora_digda_kristiawan]  1.0 1.000000  1.0    0.600000         136
    1          [fabian_ananda_merdana, muhammad_rifqi_rizqullah]  1.0 1.000000  1.0    0.633333         158
    2                     [dewita_anggraini, luthfi_triaswangga]  1.0 1.000000  1.0    0.600000         159
    3             [ashrul_rifki_ardiyhasa, tomi_martino_affandi]  1.0 1.000000  1.0    0.500000         161
    4  